In [1]:
import os

work_dir = os.getcwd()
base_dir = os.path.normpath(os.path.join(work_dir, "..", ".."))
data_dir = os.path.normpath(os.path.join(base_dir, "data", "model_comparison"))
file_name = 'results.csv'

In [2]:
import pandas as pd

df = pd.read_csv(os.path.normpath(os.path.join(data_dir, file_name)))
df.columns = df.columns.str.strip()
df_plot = df[(df['test_mae_static'] < 0.1) & (df['test_mae_transient'] < 0.1)].copy()

In [3]:
translate = {
    'linear_regression': 'Lineare \nRegression',
    'lasso_regression': 'Lasso-\nRegression',
    'static_mlp': 'MLP \n(statisch)',
    'dynamic_mlp': 'MLP \n(dynamisch)',
    'conv1d': 'CNN',
    'lstm': 'LSTM',
}
df_plot["model_key"] = df_plot["model_key"].map(translate)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(
    context="paper",        # scales fonts & lines for publications
    style="whitegrid",      # white background + subtle grid
    palette="colorblind",   # colorblind-safe categorical palette
    font="serif",           # optional: matches LaTeX look
)

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
})

mpl.use("pgf")

mpl.rcParams.update({
    "text.usetex": True,
    "pgf.texsystem": "pdflatex",   # oder "xelatex", "lualatex"
    "font.family": "serif",
    "pgf.rcfonts": False,
})

In [5]:
TEXTWIDTH_PT = 458.08954
INCH_PER_PT = 1.0 / 72.27

fig_width_in = TEXTWIDTH_PT * INCH_PER_PT

In [7]:
fig, ax = plt.subplots(figsize=(fig_width_in, fig_width_in * 0.6))

print("Transient MAE:")
# Create boxplot for test_mae_transient
sns.violinplot(data=df_plot, x='model_key', y='test_mae_transient', hue='model_key', palette='Set2', ax=ax, legend=False, order=list(translate.values()))
# sns.boxplot(data=df_plot, x='model_key', y='test_mae_transient', hue='model_key', palette='Set2', ax=ax, legend=False, order=list(translate.values()))
ax.set_ylabel('MAE')
ax.set_xlabel('Modell Architektur')
plt.tight_layout()
# plt.show()

Transient MAE:


In [8]:
print("MAE - stationary test data:")


fig, ax = plt.subplots(figsize=(fig_width_in, fig_width_in * 0.6))
ax = sns.violinplot(
    data=df_plot, 
    x='model_key', 
    y='test_mae_static',
    order=list(translate.values()),
    palette='Set2',
    hue='model_key',
    legend=False,
    cut=0,
    inner=None,
)

sns.boxplot(
    data=df_plot, 
    x='model_key', 
    y='test_mae_static',
    order=list(translate.values()),
    width=0.04,
    fill=True,
    showcaps=True,
    boxprops=dict(facecolor="#585A5D", color="#585A5D"),
    medianprops={"color": "w", "linewidth": 2},
    whiskerprops={"linewidth": 1.5, "color":"#585A5D"},
    ax=ax
)

ax.set_ylabel('MAE')
ax.set_xlabel('Modell Architektur')
plt.tight_layout()
plt.savefig(os.path.normpath(os.path.join(base_dir, "plots", "model_comparison_stationary.pgf")), transparent=True)
# plt.show()

MAE - stationary test data:


c:\git\ModelSandbox\venv_311\Lib\site-packages\seaborn\categorical.py:700: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  artists = ax.bxp(**boxplot_kws)


In [9]:
print("MAE - Transient test data:")

fig, ax = plt.subplots(figsize=(fig_width_in, fig_width_in * 0.6))
ax = sns.violinplot(
    data=df_plot, 
    x='model_key', 
    y='test_mae_transient',
    order=list(translate.values()),
    palette='Set2',
    hue='model_key',
    legend=False,
    cut=0,
    inner=None,
)

sns.boxplot(
    data=df_plot, 
    x='model_key', 
    y='test_mae_transient',
    order=list(translate.values()),
    width=0.04,
    fill=True,
    showcaps=True,
    boxprops=dict(facecolor="#585A5D", color="#585A5D"),
    medianprops={"color": "w", "linewidth": 2},
    whiskerprops={"linewidth": 1.5, "color":"#585A5D"},
    ax=ax
)

ax.set_ylabel('MAE')
ax.set_xlabel('Modell Architektur')
plt.tight_layout()
plt.savefig(os.path.normpath(os.path.join(base_dir, "plots", "model_comparison_transient.pgf")), transparent=True)
# plt.show()

c:\git\ModelSandbox\venv_311\Lib\site-packages\seaborn\categorical.py:700: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  artists = ax.bxp(**boxplot_kws)


MAE - Transient test data:


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==========================================
# 1. Daten einlesen und filtern
# ==========================================
df = pd.read_csv(os.path.normpath(os.path.join(data_dir, file_name)))
df.columns = df.columns.str.strip()

# Wir filtern starke Ausreißer heraus, um die Diagramme lesbar zu halten.
# Ein MAE > 0.08 ist für deine Top-Ergebnisse (die bei ~0.01 liegen) irrelevant.
threshold = 0.08
df_plot = df[(df['test_mae_static'] < threshold) & (df['test_mae_transient'] < threshold)].copy()

# ==========================================
# 2. Layout erstellen (2x2 Grid)
# ==========================================
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Architektur-Vergleich: Stationär vs. Transient (Bereinigt)', fontsize=18, fontweight='bold')

# ==========================================
# Plot 1: Bereinigter Trade-off Scatter Plot (Oben Links)
# ==========================================
# Keine variablen Größen mehr, nur Farbe für das Modell und Transparenz gegen Overplotting
sns.scatterplot(
    data=df_plot, 
    x='test_mae_static', 
    y='test_mae_transient', 
    hue='model_key', 
    alpha=0.5,      # Halbtransparent, um Dichte sichtbar zu machen
    s=40,           # Feste, moderate Punktgröße
    edgecolor=None, # Keine Ränder für einen weicheren Look
    ax=axes[0, 0]
)
axes[0, 0].set_title('Trade-off Cluster (Zoom auf MAE < 0.08)', fontsize=14)
axes[0, 0].set_xlabel('MAE Stationär')
axes[0, 0].set_ylabel('MAE Transient')
axes[0, 0].legend(title='Architektur')

# ==========================================
# Plot 2 & 3: Violin-Plots für die Dichteverteilung (Oben Rechts & Unten Links)
# ==========================================
# Violin-Plots zeigen, wo sich die Masse der 1500 Modelle aufhält
sns.violinplot(
    data=df_plot, 
    x='model_key', 
    y='test_mae_static', 
    inner='quartile', # Zeigt Median und Quartile als Linien im Plot
    ax=axes[0, 1]
)
axes[0, 1].set_title('Dichteverteilung: MAE Stationär', fontsize=14)
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('MAE Stationär')

sns.violinplot(
    data=df_plot, 
    x='model_key', 
    y='test_mae_transient', 
    inner='quartile',
    ax=axes[1, 0]
)
axes[1, 0].set_title('Dichteverteilung: MAE Transient', fontsize=14)
axes[1, 0].set_xlabel('Modell Architektur')
axes[1, 0].set_ylabel('MAE Transient')

# ==========================================
# Plot 4: Das absolute Potenzial (Grouped Bar Chart) (Unten Rechts)
# ==========================================
# Wir suchen für jede Architektur den jeweils besten MAE-Wert
best_stationary = df_plot.groupby('model_key')['test_mae_static'].min()
best_transient = df_plot.groupby('model_key')['test_mae_transient'].min()

# Zusammenführen für den Plot
df_best = pd.DataFrame({
    'Best MAE Stationär': best_stationary,
    'Best MAE Transient': best_transient
}).reset_index()

# Daten "schmelzen" für Seaborn Barplot
df_best_melted = df_best.melt(id_vars='model_key', var_name='Metrik', value_name='Bester MAE')

sns.barplot(
    data=df_best_melted, 
    x='model_key', 
    y='Bester MAE', 
    hue='Metrik',
    ax=axes[1, 1]
)
axes[1, 1].set_title('Das absolute Potenzial (Bestes Modell pro Architektur)', fontsize=14)
axes[1, 1].set_xlabel('Modell Architektur')
axes[1, 1].set_ylabel('Niedrigster erreichter MAE')
axes[1, 1].legend(title='')

# ==========================================
# 3. Layout optimieren und anzeigen
# ==========================================
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

C:\Users\Pascal Marijan\AppData\Local\Temp\ipykernel_20820\3753835675.py:101: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
import pandas as pd

# ==========================================
# 1. Daten einlesen
# ==========================================
# Lade die CSV-Datei
df = pd.read_csv(os.path.normpath(os.path.join(data_dir, file_name)))
df.columns = df.columns.str.strip()

# ==========================================
# 2. Beste Werte pro Modelltyp ermitteln
# ==========================================
best_stat = df.groupby('model_key')['test_mae_static'].min().reset_index()
best_trans = df.groupby('model_key')['test_mae_transient'].min().reset_index()

# Zusammenführen zu einem gemeinsamen DataFrame
df_combined = pd.merge(best_stat, best_trans, on='model_key')

# ==========================================
# 3. Absolutes Optimum finden & Abweichung berechnen
# ==========================================
abs_best_stat = df_combined['test_mae_static'].min()
abs_best_trans = df_combined['test_mae_transient'].min()

df_combined['diff_stat'] = ((df_combined['test_mae_static'] - abs_best_stat) / abs_best_stat) * 100
df_combined['diff_trans'] = ((df_combined['test_mae_transient'] - abs_best_trans) / abs_best_trans) * 100

# ==========================================
# 4. Formatierung für LaTeX
# ==========================================
# Modellnamen für LaTeX formatieren (z.B. Unterstriche escapen)
df_combined['model_key'] = df_combined['model_key'].str.replace('_', r'\_')

# Zahlen formatieren
df_combined['test_mae_static'] = df_combined['test_mae_static'].apply(lambda x: f"{x:.5f}")
df_combined['diff_stat'] = df_combined['diff_stat'].apply(lambda x: f"+{x:.1f}\\%" if x > 0 else "\\textbf{0.0\\%}")

df_combined['test_mae_transient'] = df_combined['test_mae_transient'].apply(lambda x: f"{x:.5f}")
df_combined['diff_trans'] = df_combined['diff_trans'].apply(lambda x: f"+{x:.1f}\\%" if x > 0 else "\\textbf{0.0\\%}")

# Sortieren (z.B. nach Performance auf stationären Daten)
df_combined = df_combined.sort_values('test_mae_static').reset_index(drop=True)

# ==========================================
# 5. LaTeX Code generieren und ausgeben
# ==========================================
latex_code = """\\begin{table}[htbp]
\\centering
\\caption{Vergleich der besten Modellarchitekturen auf stationären und transienten Daten. Die Abweichung ($\\Delta$) bezieht sich auf das jeweils beste Modell des Datensatzes.}
\\label{tab:model_performance}
\\begin{tabular}{l cc cc}
\\toprule
& \\multicolumn{2}{c}{\\textbf{Stationäre Daten}} & \\multicolumn{2}{c}{\\textbf{Transiente Daten}} \\\\
\\cmidrule(lr){2-3} \\cmidrule(lr){4-5}
\\textbf{Modelltyp} & \\textbf{Bester MAE} & \\textbf{$\\Delta$ Opt.} & \\textbf{Bester MAE} & \\textbf{$\\Delta$ Opt.} \\\\
\\midrule
"""

# Zeilen einfügen
for index, row in df_combined.iterrows():
    latex_code += f"{row['model_key']} & {row['test_mae_static']} & {row['diff_stat']} & {row['test_mae_transient']} & {row['diff_trans']} \\\\\n"

latex_code += """\\bottomrule
\\end{tabular}
\\end{table}"""

print(latex_code)

# ==========================================
# 6. Ergebnisse in der Konsole ausgeben
# ==========================================
print("="*60)
print("🏆 BESTE PERFORMANCE AUF STATIONÄREN DATEN 🏆")
print("="*60)
print(best_stat.to_string(index=False))
print("\n")

print("="*60)
print("🏆 BESTE PERFORMANCE AUF TRANSIENTEN DATEN 🏆")
print("="*60)
print(best_trans.to_string(index=False))
print("="*60)

\begin{table}[htbp]
\centering
\caption{Vergleich der besten Modellarchitekturen auf stationären und transienten Daten. Die Abweichung ($\Delta$) bezieht sich auf das jeweils beste Modell des Datensatzes.}
\label{tab:model_performance}
\begin{tabular}{l cc cc}
\toprule
& \multicolumn{2}{c}{\textbf{Stationäre Daten}} & \multicolumn{2}{c}{\textbf{Transiente Daten}} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5}
\textbf{Modelltyp} & \textbf{Bester MAE} & \textbf{$\Delta$ Opt.} & \textbf{Bester MAE} & \textbf{$\Delta$ Opt.} \\
\midrule
static\_mlp & 0.00745 & \textbf{0.0\%} & 0.01140 & +16.2\% \\
conv1d & 0.00822 & +10.3\% & 0.01278 & +30.3\% \\
dynamic\_mlp & 0.01017 & +36.5\% & 0.01226 & +25.0\% \\
lstm & 0.01084 & +45.4\% & 0.00981 & \textbf{0.0\%} \\
linear\_regression & 0.01400 & +87.7\% & 0.01410 & +43.7\% \\
lasso\_regression & 0.02783 & +273.3\% & 0.03205 & +226.6\% \\
\bottomrule
\end{tabular}
\end{table}
🏆 BESTE PERFORMANCE AUF STATIONÄREN DATEN 🏆
        model_key  test_mae_static
   